In [49]:
import torch
import transformer_lens
from sae_lens import SAE


In [50]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [51]:
sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.6.hook_resid_pre",
    device=device,
)

In [52]:
model = transformer_lens.HookedTransformer.from_pretrained("gpt2-small",  device=device, use_cache=True)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6456.47it/s]


Loaded pretrained model gpt2-small into HookedTransformer


In [53]:
import json

with open('story.json') as f:
    data = json.load(f)
print(len(data))

20


In [54]:
# alphas_service = [ 40 ,45, 50, 55, 60, 65, 70, 75 ]
alphas_service = [30,32,34,36,38,40,41,42,43,44,45]
alphas_classic = [1,5,10,17,18,19,20,21,22]
quantiles = [ 0.25, 0.5, 0.75, 0.8, 0.85, 0.9]
# Candidate violence features retained for later ablations.
ids = [4192, 20026, 4214, 8301, 19470, 4262, 13374, 15462, 12552]

CONCEPT_FEATURE_ID = 20026
vector = sae.W_dec[CONCEPT_FEATURE_ID].detach()

HOOK_NAME = "blocks.6.hook_resid_pre"

In [55]:
vector.shape

torch.Size([768])

In [56]:
FUNCTION_WORDS = {
    # Articles and determiners
    "a", "an", "the", "this", "that", "these", "those",
    "each", "every", "either", "neither", "some", "any",
    "all", "both", "few", "many", "much", "several",

    # Pronouns
    "i", "me", "my", "mine", "myself",
    "you", "your", "yours", "yourself", "yourselves",
    "he", "him", "his", "himself",
    "she", "her", "hers", "herself",
    "it", "its", "itself",
    "we", "us", "our", "ours", "ourselves",
    "they", "them", "their", "theirs", "themselves",
    "who", "whom", "whose", "which", "what",

    # Prepositions
    "of", "to", "in", "for", "on", "with", "at", "by",
    "from", "up", "about", "into", "over", "after",
    "beneath", "under", "above", "through", "during",
    "before", "between", "without", "within", "along",
    "across", "behind", "beyond", "toward", "towards",
    "against", "among", "around", "near",

    # Conjunctions
    "and", "but", "or", "nor", "so", "yet",
    "if", "because", "although", "though", "while",
    "when", "whenever", "where", "whereas", "whether",
    "unless", "until", "since", "than", "as",

    # Auxiliary verbs
    "am", "is", "are", "was", "were",
    "be", "been", "being",
    "have", "has", "had", "having",
    "do", "does", "did", "doing",
    "will", "would", "shall", "should",
    "can", "could", "may", "might", "must",

    # Particles and function-like adverbs
    "not", "no", "yes", "very", "too", "also",
    "only", "just", "even", "still", "already",
}


def build_token_sets(tokenizer):
    content_ids = []
    function_ids = []

    for token_id in range(len(tokenizer)):
        token_text = tokenizer.decode(
            [token_id],
            clean_up_tokenization_spaces=False
        )

        # Берём только GPT-2-токены, представляющие целое
        # слово с начальным пробелом.
        if not token_text.startswith(" "):
            continue

        word = token_text.strip().lower()

        # Исключаем числа, пунктуацию и BPE-фрагменты
        if not word.isalpha():
            continue

        if word in FUNCTION_WORDS:
            function_ids.append(token_id)

        elif len(word) >= 2:
            content_ids.append(token_id)

    return (
        torch.tensor(content_ids, dtype=torch.long),
        torch.tensor(function_ids, dtype=torch.long)
    )


content_ids, function_ids = build_token_sets(
    model.tokenizer
)

content_ids = content_ids.to(model.W_U.device)
function_ids = function_ids.to(model.W_U.device)

print("Content token count:", len(content_ids))
print("Function token count:", len(function_ids))

Content token count: 31682
Function token count: 357


In [57]:
model_device = model.W_U.device
model_dtype = model.W_U.dtype
sae_device = sae.W_dec.device
sae_dtype = sae.W_dec.dtype

W_U = model.W_U
W_dec = sae.W_dec.to(device=model_device, dtype=model_dtype)

content_ids = content_ids.to(model_device)
function_ids = function_ids.to(model_device)

with torch.inference_mode():
    content_direction = W_U[:, content_ids].mean(dim=-1)
    function_direction = W_U[:, function_ids].mean(dim=-1)
    content_contrast_direction = (
        content_direction - function_direction
    ).detach()

    # Exact all-feature SAE score weights.
    feature_content_scores = (
        W_dec @ content_contrast_direction
    ).detach()

# Constants used by the exact SAE hook.
HOOK_FEATURE_SCORES = feature_content_scores.to(
    device=sae_device,
    dtype=sae_dtype,
).reshape(1, 1, -1)

# Constants used by the fast residual approximation.
HOOK_CONTRAST = content_contrast_direction.to(
    device=model_device,
    dtype=model_dtype,
).reshape(1, 1, -1)

HOOK_B_DEC = sae.b_dec.to(
    device=model_device,
    dtype=model_dtype,
).reshape(1, 1, -1)

HOOK_STEERING_VECTOR = vector.to(
    device=model_device,
    dtype=model_dtype,
).reshape(1, 1, -1)

print("feature_content_scores:", feature_content_scores.shape)

feature_content_scores: torch.Size([24576])


In [58]:
CONTENT_THRESHOLD_EXACT = 6.536377
GATE_QUANTILE = 0.5

# Request the fast residual approximation. It is enabled only if calibration
# shows that it faithfully reproduces the original SAE gate.
REQUEST_FAST_GATE = True

gate_score_buffer = []


def reset_gate_stats():
    gate_score_buffer.clear()


def read_gate_stats(threshold):
    """Synchronize once after generation, not once per generated token."""
    if not gate_score_buffer:
        return {
            "calls": 0,
            "triggered": 0,
            "trigger_rate": 0.0,
            "score_min": None,
            "score_mean": None,
            "score_max": None,
        }

    scores = torch.cat([
        score.detach().reshape(-1)
        for score in gate_score_buffer
    ]).float().cpu()

    triggered = int((scores >= threshold).sum().item())
    calls = scores.numel()

    return {
        "calls": calls,
        "triggered": triggered,
        "trigger_rate": triggered / max(calls, 1),
        "score_min": scores.min().item(),
        "score_mean": scores.mean().item(),
        "score_max": scores.max().item(),
    }


def hook_steering_classic(tensor, hook):
    """Original SAE content gate, optimized to avoid per-token CPU syncs."""
    last_tensor = tensor[:, -1:, :]

    sae_input = last_tensor.to(
        device=sae_device,
        dtype=sae_dtype,
    )

    steering_vector = HOOK_STEERING_VECTOR.to(
        device=tensor.device,
        dtype=tensor.dtype,
    )
    steered_last = last_tensor + alpha_classic * steering_vector

    # Appending a CUDA tensor is asynchronous. Do not call item/cpu/tolist here.

    result = tensor.clone()
    result[:, -1:, :] = steered_last
    return result


def hook_steering_fast(tensor, hook):
    """
    Fast approximation:
        z @ (W_dec @ contrast) ~= (h - b_dec) @ contrast.
    It replaces the 768->24576 SAE encoder with one 768-dimensional dot product.
    """
    last_tensor = tensor[:, -1:, :]

    contrast = HOOK_CONTRAST.to(
        device=tensor.device,
        dtype=tensor.dtype,
    )
    b_dec = HOOK_B_DEC.to(
        device=tensor.device,
        dtype=tensor.dtype,
    )

    content_score = (
        (last_tensor - b_dec) * contrast
    ).sum(dim=-1, keepdim=True)

    gate = (
        content_score >= FAST_THRESHOLD
    ).to(tensor.dtype)

    steering_vector = HOOK_STEERING_VECTOR.to(
        device=tensor.device,
        dtype=tensor.dtype,
    )
    steered_last = last_tensor + gate * alpha_service * steering_vector

    gate_score_buffer.append(content_score.detach())

    result = tensor.clone()
    result[:, -1:, :] = steered_last
    return result

import torch


# ---------------------------
# Гиперпараметры
# ---------------------------

gate_rate = 0.40
max_new_tokens = 120

generation_seed = 42
gate_seed = 10_000 + generation_seed


# ---------------------------
# Случайная маска
# ---------------------------

# Отдельный генератор, чтобы не менять RNG семплирования текста
gate_rng = torch.Generator(device="cpu")
gate_rng.manual_seed(gate_seed)

n_triggered = round(max_new_tokens * gate_rate)

gate_schedule = torch.zeros(
    max_new_tokens,
    dtype=torch.bool,
)

random_positions = torch.randperm(
    max_new_tokens,
    generator=gate_rng,
)[:n_triggered]

gate_schedule[random_positions] = True


# Состояние текущей генерации
gate_state = {
    "step": 0,
    "triggered": 0,
}


# ---------------------------
# Хук
# ---------------------------

def hook_random_steering(tensor, hook):
    step = gate_state["step"]
    gate_state["step"] += 1

    # Защита от дополнительных вызовов
    if step >= len(gate_schedule):
        return tensor

    if not gate_schedule[step]:
        return tensor

    gate_state["triggered"] += 1

    steering_vector = vector.detach().to(
        device=tensor.device,
        dtype=tensor.dtype,
    ).reshape(-1)

    if steering_vector.numel() != tensor.shape[-1]:
        raise ValueError(
            f"vector size={steering_vector.numel()}, "
            f"d_model={tensor.shape[-1]}"
        )

    result = tensor.clone()

    # [1, 768] + [1, 768]
    result[:, -1, :] = (
        tensor[:, -1, :]
        + alpha_random * steering_vector.unsqueeze(0)
    )

    return result


@torch.inference_mode()
def calibrate_fast_gate(texts, quantile=0.90, max_texts=20):
    """Compare the exact and fast scores once on ordinary prompt states."""
    exact_parts = []
    fast_parts = []

    for text in texts[:max_texts]:
        _, cache = model.run_with_cache(
            text,
            names_filter=lambda name: name == HOOK_NAME,
        )
        hidden = cache[HOOK_NAME]

        sae_acts = sae.encode(hidden.to(
            device=sae_device,
            dtype=sae_dtype,
        ))
        exact = (
            sae_acts * HOOK_FEATURE_SCORES
        ).sum(dim=-1).reshape(-1)

        fast = (
            (hidden - HOOK_B_DEC) * HOOK_CONTRAST
        ).sum(dim=-1).reshape(-1)

        exact_parts.append(exact.float().to(model_device))
        fast_parts.append(fast.float().to(model_device))

        del cache

    exact_scores = torch.cat(exact_parts)
    fast_scores = torch.cat(fast_parts)

    exact_threshold = torch.quantile(exact_scores, quantile)
    fast_threshold = torch.quantile(fast_scores, quantile)

    correlation = torch.corrcoef(torch.stack([
        exact_scores,
        fast_scores,
    ]))[0, 1]

    agreement = (
        (exact_scores >= exact_threshold)
        == (fast_scores >= fast_threshold)
    ).float().mean()

    return {
        "exact_threshold": exact_threshold.item(),
        "fast_threshold": fast_threshold.item(),
        "correlation": correlation.item(),
        "gate_agreement": agreement.item(),
        "positions": exact_scores.numel(),
    }


fast_calibration = calibrate_fast_gate(
    data,
    quantile=GATE_QUANTILE,
)
FAST_THRESHOLD = fast_calibration["fast_threshold"]
USE_FAST_GATE = (
    REQUEST_FAST_GATE
    and fast_calibration["correlation"] >= 0.90
    and fast_calibration["gate_agreement"] >= 0.85
)

ACTIVE_THRESHOLD = (
    FAST_THRESHOLD
    if USE_FAST_GATE
    else CONTENT_THRESHOLD_EXACT
)

print("Fast-gate calibration:", fast_calibration)
print("Using:", "fast gate" if USE_FAST_GATE else "exact SAE gate")
print("Active threshold:", ACTIVE_THRESHOLD)

Fast-gate calibration: {'exact_threshold': 0.3624267876148224, 'fast_threshold': 0.5824456810951233, 'correlation': 0.9936432838439941, 'gate_agreement': 0.9296636581420898, 'positions': 1308}
Using: fast gate
Active threshold: 0.5824456810951233


In [59]:
import os
import time

OUTPUT_DIR = "res_sae_service/fast_gate"
MAX_PROMPTS = 20
NUM_SEEDS = 5
MAX_NEW_TOKENS = 120

os.makedirs(OUTPUT_DIR, exist_ok=True)


def synchronize_model_device():
    # One synchronization per complete generation gives correct timings
    # without reintroducing per-token synchronization overhead.
    if model_device.type == "cuda":
        torch.cuda.synchronize(model_device)

for idx_p, prompt in enumerate(data[:MAX_PROMPTS]):
    results_by_alpha = {alpha_value: [] for alpha_value in alphas_service}
    print(f"prompt: {idx_p}")

    for i in range(NUM_SEEDS):
        seed = 42 + i + idx_p

        # Baseline is independent of alpha, so compute it only once.
        torch.manual_seed(seed)
        synchronize_model_device()
        baseline_start = time.perf_counter()
        no_steering = model.generate(
            prompt,
            temperature=0.9,
            max_new_tokens=MAX_NEW_TOKENS,
            top_p=0.9,
            top_k=50,
            use_past_kv_cache=True,
            verbose=False,
        )
        synchronize_model_device()
        baseline_seconds = time.perf_counter() - baseline_start

        for alpha_service in alphas_service:
            reset_gate_stats()
            torch.manual_seed(seed)

            model.add_hook(
                name=HOOK_NAME,
                hook=hook_steering_fast,
                dir="fwd",
            )

            synchronize_model_device()
            steering_start = time.perf_counter()
            try:
                steering = model.generate(
                    prompt,
                    temperature=0.9,
                    max_new_tokens=MAX_NEW_TOKENS,
                    top_p=0.9,
                    top_k=50,
                    use_past_kv_cache=True,
                    verbose=False,
                )
            finally:
                model.reset_hooks()

            synchronize_model_device()
            steering_seconds = time.perf_counter() - steering_start
            gate_diagnostics = read_gate_stats(ACTIVE_THRESHOLD)

            results_by_alpha[alpha_service].append({
                "prompt": prompt,
                "no_steering": no_steering,
                "steering": steering,
                "alpha": alpha_service,
                "seed": seed,
                "q": GATE_QUANTILE
            })

            print(
                f"  alpha={alpha_service}: "
                f"trigger_rate={gate_diagnostics['trigger_rate']:.1%}, "
                f"slowdown={steering_seconds / max(baseline_seconds, 1e-9):.2f}x"
            )

    for alpha_value, results in results_by_alpha.items():
        output_path = os.path.join(
            OUTPUT_DIR,
            f"{idx_p}_{alpha_value}.json",
        )
        with open(output_path, mode="w", encoding="utf-8") as file:
            json.dump(results, file, indent=2, ensure_ascii=False)

prompt: 0
  alpha=30: trigger_rate=35.0%, slowdown=1.29x
  alpha=32: trigger_rate=35.0%, slowdown=2.10x
  alpha=34: trigger_rate=35.0%, slowdown=2.09x
  alpha=36: trigger_rate=35.0%, slowdown=2.64x
  alpha=38: trigger_rate=35.0%, slowdown=1.12x
  alpha=40: trigger_rate=29.2%, slowdown=1.03x
  alpha=41: trigger_rate=33.3%, slowdown=1.01x
  alpha=42: trigger_rate=33.3%, slowdown=1.00x
  alpha=43: trigger_rate=30.8%, slowdown=1.01x
  alpha=44: trigger_rate=40.8%, slowdown=1.02x
  alpha=45: trigger_rate=40.8%, slowdown=1.01x
  alpha=30: trigger_rate=37.5%, slowdown=1.01x
  alpha=32: trigger_rate=38.3%, slowdown=1.02x
  alpha=34: trigger_rate=37.5%, slowdown=1.02x
  alpha=36: trigger_rate=31.7%, slowdown=1.02x
  alpha=38: trigger_rate=31.7%, slowdown=1.02x
  alpha=40: trigger_rate=32.5%, slowdown=1.02x
  alpha=41: trigger_rate=25.8%, slowdown=1.03x
  alpha=42: trigger_rate=24.2%, slowdown=1.02x
  alpha=43: trigger_rate=24.2%, slowdown=1.02x
  alpha=44: trigger_rate=44.2%, slowdown=1.03x
  a

In [63]:
import os
import time

OUTPUT_DIR = "res_sae_service/classic"
MAX_PROMPTS = 20
NUM_SEEDS = 5
MAX_NEW_TOKENS = 120

os.makedirs(OUTPUT_DIR, exist_ok=True)


def synchronize_model_device():
    # One synchronization per complete generation gives correct timings
    # without reintroducing per-token synchronization overhead.
    if model_device.type == "cuda":
        torch.cuda.synchronize(model_device)

for idx_p, prompt in enumerate(data[:MAX_PROMPTS]):
    results_by_alpha = {alpha_value: [] for alpha_value in alphas_classic}
    print(f"prompt: {idx_p}")

    for i in range(NUM_SEEDS):
        seed = 42 + i + idx_p

        # Baseline is independent of alpha, so compute it only once.
        torch.manual_seed(seed)
        synchronize_model_device()
        baseline_start = time.perf_counter()
        no_steering = model.generate(
            prompt,
            temperature=0.9,
            max_new_tokens=MAX_NEW_TOKENS,
            top_p=0.9,
            top_k=50,
            use_past_kv_cache=True,
            verbose=False,
        )
        synchronize_model_device()
        baseline_seconds = time.perf_counter() - baseline_start

        for alpha_classic in alphas_classic:
            reset_gate_stats()
            torch.manual_seed(seed)

            model.add_hook(
                name=HOOK_NAME,
                hook=hook_steering_classic,
                dir="fwd",
            )

            synchronize_model_device()
            steering_start = time.perf_counter()
            try:
                steering = model.generate(
                    prompt,
                    temperature=0.9,
                    max_new_tokens=MAX_NEW_TOKENS,
                    top_p=0.9,
                    top_k=50,
                    use_past_kv_cache=True,
                    verbose=False,
                )
            finally:
                model.reset_hooks()

            synchronize_model_device()
            steering_seconds = time.perf_counter() - steering_start
            gate_diagnostics = read_gate_stats(ACTIVE_THRESHOLD)

            results_by_alpha[alpha_classic].append({
                "prompt": prompt,
                "no_steering": no_steering,
                "steering": steering,
                "alpha": alpha_classic,
                "seed": seed,
            })

            print(
                f"  alpha={alpha_classic}: "
                f"trigger_rate={gate_diagnostics['trigger_rate']:.1%}, "
                f"slowdown={steering_seconds / max(baseline_seconds, 1e-9):.2f}x"
            )

    for alpha_value, results in results_by_alpha.items():
        output_path = os.path.join(
            OUTPUT_DIR,
            f"{idx_p}_{alpha_value}.json",
        )
        with open(output_path, mode="w", encoding="utf-8") as file:
            json.dump(results, file, indent=2, ensure_ascii=False)

prompt: 0
  alpha=1: trigger_rate=0.0%, slowdown=0.90x
  alpha=5: trigger_rate=0.0%, slowdown=0.90x
  alpha=10: trigger_rate=0.0%, slowdown=0.90x
  alpha=17: trigger_rate=0.0%, slowdown=0.89x
  alpha=18: trigger_rate=0.0%, slowdown=0.90x
  alpha=19: trigger_rate=0.0%, slowdown=0.90x
  alpha=20: trigger_rate=0.0%, slowdown=0.89x
  alpha=21: trigger_rate=0.0%, slowdown=0.90x
  alpha=22: trigger_rate=0.0%, slowdown=0.89x
  alpha=1: trigger_rate=0.0%, slowdown=1.00x
  alpha=5: trigger_rate=0.0%, slowdown=1.01x
  alpha=10: trigger_rate=0.0%, slowdown=1.02x
  alpha=17: trigger_rate=0.0%, slowdown=1.02x
  alpha=18: trigger_rate=0.0%, slowdown=1.02x
  alpha=19: trigger_rate=0.0%, slowdown=1.02x
  alpha=20: trigger_rate=0.0%, slowdown=1.01x
  alpha=21: trigger_rate=0.0%, slowdown=1.01x
  alpha=22: trigger_rate=0.0%, slowdown=1.01x
  alpha=1: trigger_rate=0.0%, slowdown=1.01x
  alpha=5: trigger_rate=0.0%, slowdown=1.00x
  alpha=10: trigger_rate=0.0%, slowdown=1.00x
  alpha=17: trigger_rate=0.0%,

KeyboardInterrupt: 

In [65]:
import os
import time

OUTPUT_DIR = "res_sae_service/random"
MAX_PROMPTS = 20
NUM_SEEDS = 5
MAX_NEW_TOKENS = 120

os.makedirs(OUTPUT_DIR, exist_ok=True)


def synchronize_model_device():
    # One synchronization per complete generation gives correct timings
    # without reintroducing per-token synchronization overhead.
    if model_device.type == "cuda":
        torch.cuda.synchronize(model_device)

for idx_p, prompt in enumerate(data[:MAX_PROMPTS]):
    results_by_alpha = {alpha_value: [] for alpha_value in alphas_classic}
    print(f"prompt: {idx_p}")

    for i in range(NUM_SEEDS):
        seed = 42 + i + idx_p

        # Baseline is independent of alpha, so compute it only once.
        torch.manual_seed(seed)
        synchronize_model_device()
        baseline_start = time.perf_counter()
        no_steering = model.generate(
            prompt,
            temperature=0.9,
            max_new_tokens=MAX_NEW_TOKENS,
            top_p=0.9,
            top_k=50,
            use_past_kv_cache=True,
            verbose=False,
        )
        synchronize_model_device()
        baseline_seconds = time.perf_counter() - baseline_start

        for alpha_random in alphas_classic:
            reset_gate_stats()
            torch.manual_seed(seed)

            model.add_hook(
                name=HOOK_NAME,
                hook=hook_random_steering,
                dir="fwd",
            )

            synchronize_model_device()
            steering_start = time.perf_counter()
            try:
                steering = model.generate(
                    prompt,
                    temperature=0.9,
                    max_new_tokens=MAX_NEW_TOKENS,
                    top_p=0.9,
                    top_k=50,
                    use_past_kv_cache=True,
                    verbose=False,
                )
            finally:
                model.reset_hooks()

            synchronize_model_device()
            steering_seconds = time.perf_counter() - steering_start
            gate_diagnostics = read_gate_stats(ACTIVE_THRESHOLD)

            results_by_alpha[alpha_random].append({
                "prompt": prompt,
                "no_steering": no_steering,
                "steering": steering,
                "alpha": alpha_random,
                "seed": seed,
            })

            print(
                f"  alpha={alpha_random}: "
                f"trigger_rate={gate_diagnostics['trigger_rate']:.1%}, "
                f"slowdown={steering_seconds / max(baseline_seconds, 1e-9):.2f}x"
            )

    for alpha_value, results in results_by_alpha.items():
        output_path = os.path.join(
            OUTPUT_DIR,
            f"{idx_p}_{alpha_value}.json",
        )
        with open(output_path, mode="w", encoding="utf-8") as file:
            json.dump(results, file, indent=2, ensure_ascii=False)

prompt: 0
  alpha=1: trigger_rate=0.0%, slowdown=0.89x
  alpha=5: trigger_rate=0.0%, slowdown=0.90x
  alpha=10: trigger_rate=0.0%, slowdown=0.90x
  alpha=17: trigger_rate=0.0%, slowdown=0.90x
  alpha=18: trigger_rate=0.0%, slowdown=0.90x
  alpha=19: trigger_rate=0.0%, slowdown=0.89x
  alpha=20: trigger_rate=0.0%, slowdown=0.90x
  alpha=21: trigger_rate=0.0%, slowdown=0.90x
  alpha=22: trigger_rate=0.0%, slowdown=0.90x
  alpha=1: trigger_rate=0.0%, slowdown=1.01x
  alpha=5: trigger_rate=0.0%, slowdown=1.01x
  alpha=10: trigger_rate=0.0%, slowdown=1.00x
  alpha=17: trigger_rate=0.0%, slowdown=1.01x
  alpha=18: trigger_rate=0.0%, slowdown=1.01x
  alpha=19: trigger_rate=0.0%, slowdown=1.01x
  alpha=20: trigger_rate=0.0%, slowdown=1.00x
  alpha=21: trigger_rate=0.0%, slowdown=1.01x
  alpha=22: trigger_rate=0.0%, slowdown=1.00x
  alpha=1: trigger_rate=0.0%, slowdown=1.00x
  alpha=5: trigger_rate=0.0%, slowdown=0.99x
  alpha=10: trigger_rate=0.0%, slowdown=1.00x
  alpha=17: trigger_rate=0.0%,

In [ ]:
service_ids = [
    8565, 9397, 10180,
    8438, 11490, 9857, 10992, 12086, 10268,
    10797, 11738, 11867, 8103, 838, 12368,
    9439, 10255, 9622, 8207, 15190,
    19090, 2175, 20589,
    9991, 9438, 16376, 4009, 16358, 9618, 12046
]

service_tensor = torch.tensor(
    service_ids,
    device=device,
    dtype=torch.long
)

W_dec = sae.W_dec.to(
    device=device,
    dtype=model.W_U.dtype
)

W_U = model.W_U

# [number_of_service_features, vocabulary]
selected_logit_effects = (
    W_dec[service_tensor] @ W_U
)

# Нормализация внутри каждого feature
centered_effects = (
    selected_logit_effects
    - selected_logit_effects.mean(dim=-1, keepdim=True)
)

normalized_effects = (
    centered_effects
    / centered_effects.std(
        dim=-1,
        keepdim=True
    ).clamp_min(1e-8)
)

content_enrichment = normalized_effects[
    :, content_ids
].mean(dim=-1)

function_enrichment = normalized_effects[
    :, function_ids
].mean(dim=-1)

prediction_score = (
    content_enrichment
    - function_enrichment
)